In [192]:
import pandas as pd
import numpy as np

In [193]:
df = pd.read_excel('SPY - Daily Closing Prices through Dec 2025.xlsx')
df= df.dropna(how='all').reset_index(drop=True)
df.columns = df.iloc[0]
df = df[1:].reset_index(drop=True)
df.columns = ['Date', 'SPY']
df['Date'] = pd.to_datetime(df['Date'])

In [194]:
ff = pd.read_csv('FEDFUNDS.csv', parse_dates=['observation_date'])
ff['month'] = ff['observation_date'].dt.to_period('M')

In [195]:
df['month'] = df['Date'].dt.to_period('M')
df = df.merge(ff[['month', 'FEDFUNDS']], on = 'month', how = 'left')
df['rfr_daily'] = df['FEDFUNDS']/100/252

In [196]:
df['SPY'] = pd.to_numeric(df["SPY"], errors = "coerce")
df["daily_return"] = df["SPY"].pct_change()

In [197]:
df["excess_return"] = df["daily_return"] - df["rfr_daily"]

In [198]:
df

,Date,SPY,month,FEDFUNDS,rfr_daily,daily_return,excess_return
0,1993-01-29,24.313,1993-01,3.02,0.000120,NaN,NaN
1,1993-02-01,24.486,1993-02,3.03,0.000120,0.007116,0.006995
2,1993-02-02,24.538,1993-02,3.03,0.000120,0.002124,0.002003
3,1993-02-03,24.797,1993-02,3.03,0.000120,0.010555,0.010435
4,1993-02-04,24.901,1993-02,3.03,0.000120,0.004194,0.004074
...,...,...,...,...,...,...,...
8265,2025-11-28,683.680,2025-11,3.88,0.000154,0.005959,0.005805
8266,2025-12-01,680.270,2025-12,3.72,0.000148,-0.004988,-0.005135
8267,2025-12-02,681.480,2025-12,3.72,0.000148,0.001779,0.001631
8268,2025-12-03,683.810,2025-12,3.72,0.000148,0.003419,0.003271


In [199]:
df["Down_Flag"] = (df['daily_return'] < 0).astype(int)

In [200]:
df

,Date,SPY,month,FEDFUNDS,rfr_daily,daily_return,excess_return,Down_Flag
0,1993-01-29,24.313,1993-01,3.02,0.000120,NaN,NaN,0
1,1993-02-01,24.486,1993-02,3.03,0.000120,0.007116,0.006995,0
2,1993-02-02,24.538,1993-02,3.03,0.000120,0.002124,0.002003,0
3,1993-02-03,24.797,1993-02,3.03,0.000120,0.010555,0.010435,0
4,1993-02-04,24.901,1993-02,3.03,0.000120,0.004194,0.004074,0
...,...,...,...,...,...,...,...,...
8265,2025-11-28,683.680,2025-11,3.88,0.000154,0.005959,0.005805,0
8266,2025-12-01,680.270,2025-12,3.72,0.000148,-0.004988,-0.005135,1
8267,2025-12-02,681.480,2025-12,3.72,0.000148,0.001779,0.001631,0
8268,2025-12-03,683.810,2025-12,3.72,0.000148,0.003419,0.003271,0


In [205]:
rows = []
for n in [1, 2, 3, 4, 5, 6]:
    down = df["Down_Flag"].rolling(n).sum().eq(n).astype(int)
    pos  = down.shift(1)
    strat = df["excess_return"] * pos
    ann_ret = strat.mean() * 252
    ann_vol = strat.std() * np.sqrt(252)
    rows.append({"N": n, "ann_ret": ann_ret, "ann_vol": ann_vol,
                 "sharpe": ann_ret / ann_vol, "pct_days_in": pos.mean()})
print(pd.DataFrame(rows))

   N   ann_ret   ann_vol    sharpe  pct_days_in
0  1  0.083050  0.139505  0.595316     0.455799
1  2  0.061685  0.104742  0.588919     0.200024
2  3  0.050243  0.074579  0.673679     0.085742
3  4  0.030683  0.055380  0.554039     0.033740
4  5  0.019862  0.038381  0.517486     0.012577
5  6  0.010037  0.030860  0.325235     0.003870


In [204]:
# buy and hold
sharpe_bnh = df["excess_return"].mean()*252 / (df["excess_return"].std()*np.sqrt(252))
print("buy and hold - sharpe -", sharpe_bnh)

buy and hold - sharpe - 0.5018404933695845
